# 02 — Preprocessing & Splitting

**Insurance Claim Fraud Detection & Action Recommendation System**

This is the most leakage-sensitive notebook in the project. It turns Notebook
01's documented findings into concrete, auditable preprocessing decisions,
then splits the data **before** fitting anything — every transformer used
downstream is fit on the training split only.

**What this notebook does:**
- Step 0 — reads Notebook 01's findings and re-verifies the numbers behind
  each column decision
- Step 1 — drops identifier / redundant columns (with reasons)
- Step 2 — fixes the two documented data-quality issues (differently, on
  purpose)
- Step 3 — defines an ordinal / one-hot encoding plan (not yet fit)
- Step 4 — splits the cleaned, **unencoded** data into a primary stratified
  train/val/test and a secondary temporal robustness split, then fits the
  encoding pipeline on the primary training split only
- Step 5 — saves the cleaned data, both splits, the fitted pipeline, and an
  audit manifest to Drive
- Step 6 — runs structural assertions that must pass before this notebook is
  considered done

**Explicitly out of scope here:** model training, resampling/SMOTE, scaling,
hyperparameter tuning, calibration, thresholding. Those belong to
`03_baseline.ipynb` onward. This notebook does not touch the test set beyond
creating it, transforming it with the train-fit pipeline, and saving it.

## 1. Environment & Google Drive Setup

Same conventions as Notebook 01: everything is derived from one
`PROJECT_ROOT`, Drive is mounted before any path is touched, and every
expected input is verified to exist before this notebook proceeds — including
Notebook 01's own output, since Step 0 depends on it.

In [1]:
"""Notebook 02 - Preprocessing & Splitting: imports and shared configuration."""
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

TARGET_COL = "FraudFound_P"
RANDOM_STATE = 42

saved_files = []  # every artifact written to Drive is recorded here for the final recap
print("Libraries imported.")

Libraries imported.


### Mount Google Drive

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "This notebook is designed to run in Google Colab with the team's shared "
        "Drive mounted. `google.colab` is not available in this environment - open "
        "the notebook in Colab and re-run."
    ) from exc

Mounted at /content/drive


### Define project paths

Same layout as Notebook 01, plus `models/` for the fitted preprocessing
pipeline and `data/processed/` for everything this notebook produces.

In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/InsuranceFraudProject")

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_EDA_DIR = RESULTS_DIR / "figures" / "eda"
MODELS_DIR = PROJECT_ROOT / "models"

RAW_DATA_PATH = DATA_RAW_DIR / "fraud_oracle.csv"
SIGNAL_RANKING_PATH = FIGURES_EDA_DIR / "feature_fraud_signal_ranking.csv"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project configuration")
print("-" * 60)
print(f"PROJECT_ROOT         : {PROJECT_ROOT}")
print(f"RAW_DATA_PATH         : {RAW_DATA_PATH}")
print(f"SIGNAL_RANKING_PATH   : {SIGNAL_RANKING_PATH}")
print(f"DATA_PROCESSED_DIR    : {DATA_PROCESSED_DIR}")
print(f"MODELS_DIR            : {MODELS_DIR}")

Project configuration
------------------------------------------------------------
PROJECT_ROOT         : /content/drive/MyDrive/InsuranceFraudProject
RAW_DATA_PATH         : /content/drive/MyDrive/InsuranceFraudProject/data/raw/fraud_oracle.csv
SIGNAL_RANKING_PATH   : /content/drive/MyDrive/InsuranceFraudProject/results/figures/eda/feature_fraud_signal_ranking.csv
DATA_PROCESSED_DIR    : /content/drive/MyDrive/InsuranceFraudProject/data/processed
MODELS_DIR            : /content/drive/MyDrive/InsuranceFraudProject/models


### Verify expected inputs exist

Two inputs are required: the raw dataset, and Notebook 01's feature-signal
ranking (Step 0 depends on it). Fail clearly, before doing any work, if
either is missing.

In [4]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the raw dataset at:\n  {RAW_DATA_PATH}\n\n"
        "Checklist:\n"
        "  1. Confirm Google Drive is mounted (the previous cell ran without error).\n"
        "  2. Confirm the shared Drive folder 'InsuranceFraudProject' has been added "
        "to 'My Drive', not just visible under 'Shared with me'.\n"
        "  3. Confirm the file lives at data/raw/fraud_oracle.csv inside that folder."
    )

if not SIGNAL_RANKING_PATH.exists():
    raise FileNotFoundError(
        f"Could not find Notebook 01's feature-signal ranking at:\n  {SIGNAL_RANKING_PATH}\n\n"
        "Run 01_eda_and_statistics.ipynb first - it produces this file in Section "
        "7.5.4. Step 0 of this notebook depends on it to cross-check column decisions."
    )

print("Found raw dataset and Notebook 01's feature-signal ranking.")

Found raw dataset and Notebook 01's feature-signal ranking.


## Step 0 — Read Notebook 01's Findings

Every column decision in Step 1 rests on a specific number from Notebook 01.
Some of those numbers live in a saved artifact (`feature_fraud_signal_ranking.csv`)
and are loaded directly here. Others — the redundancy checks (`PolicyType`
vs. its components, `AgeOfPolicyHolder` vs. `Age`) and the `PolicyNumber`/`Year`
correlation — were only printed/plotted in Notebook 01, not persisted as
numbers. Rather than trust those blindly, they're cheaply recomputed here
directly from the raw data as a live "trust but verify" check, using the same
method Notebook 01 used (a bias-corrected Cramér's V). This is not a redo of
EDA — it's a targeted re-check of the four numbers that directly gate a
column-drop decision below.

In [5]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Bias-corrected Cramer's V (Bergsma, 2013) - same formula as Notebook 01
    Section 7.5.1, reimplemented here since this notebook must be runnable on its
    own kernel without depending on Notebook 01's in-memory state."""
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    phi2 = chi2 / n
    phi2_corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    return float(np.sqrt(phi2_corr / denom)) if denom > 0 else np.nan


raw_df = pd.read_csv(RAW_DATA_PATH)
signal_ranking = pd.read_csv(SIGNAL_RANKING_PATH)

print(f"Loaded raw data: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print(f"Loaded signal ranking: {len(signal_ranking)} features")

Loaded raw data: 15,420 rows x 33 columns
Loaded signal ranking: 30 features


### Live re-verification of the redundancy / identifier claims

In [6]:
policynumber_year_corr = raw_df["PolicyNumber"].corr(raw_df["Year"])

policytype_reconstructed = raw_df["VehicleCategory"] + " - " + raw_df["BasePolicy"]
policytype_v = cramers_v(raw_df["PolicyType"], policytype_reconstructed)

age_policyholder_v = cramers_v(raw_df["AgeOfPolicyHolder"], raw_df["Age"])

print(f"corr(PolicyNumber, Year)                       = {policynumber_year_corr:.3f}")
print(f"Cramer's V(PolicyType, VehicleCategory-BasePolicy) = {policytype_v:.3f}")
print(f"Cramer's V(AgeOfPolicyHolder, Age)               = {age_policyholder_v:.3f}")

# Soft thresholds, not exact-match asserts: the point is to catch a real regression
# (e.g. the redundancy no longer holding), not to demand bit-for-bit reproduction of
# Notebook 01's numbers on a possibly-updated raw file.
assert policynumber_year_corr > 0.85, (
    "PolicyNumber no longer looks like a disguised timestamp - re-examine the "
    "drop-PolicyNumber decision before proceeding."
)
assert policytype_v > 0.9, (
    "PolicyType no longer looks redundant with VehicleCategory + BasePolicy - "
    "re-examine the drop-PolicyType decision before proceeding."
)
assert age_policyholder_v > 0.9, (
    "AgeOfPolicyHolder no longer looks redundant with Age - re-examine the "
    "drop-AgeOfPolicyHolder decision before proceeding."
)
print("\nAll three redundancy/identifier checks confirmed the Notebook 01 findings.")

corr(PolicyNumber, Year)                       = 0.937
Cramer's V(PolicyType, VehicleCategory-BasePolicy) = 1.000
Cramer's V(AgeOfPolicyHolder, Age)               = 0.998

All three redundancy/identifier checks confirmed the Notebook 01 findings.


### Cross-check the signal ranking for `RepNumber` and the low-signal "keep" list

In [7]:
def lookup_signal(feature: str) -> pd.Series:
    """Look up one feature's row in the signal ranking; raise clearly if it's missing
    rather than silently returning nothing (the ranking is expected to be exhaustive
    over the columns Notebook 01 tested)."""
    row = signal_ranking.loc[signal_ranking["feature"] == feature]
    if row.empty:
        raise KeyError(
            f"'{feature}' not found in {SIGNAL_RANKING_PATH.name} - check that "
            "Notebook 01 hasn't changed which features it tests."
        )
    return row.iloc[0]


rep_number_row = lookup_signal("RepNumber")
print(f"RepNumber vs. target: effect_size={rep_number_row['effect_size']:.3f}, "
      f"p_value={rep_number_row['p_value']:.3f} -> supports dropping it (no signal).")

print("\nLow-signal features kept anyway (univariate weakness != uselessness):")
for feat in ["DriverRating", "Sex", "WeekOfMonth"]:
    row = lookup_signal(feat)
    print(f"  {feat:<14} effect_size={row['effect_size']:.3f}, p_value={row['p_value']:.3f}")

RepNumber vs. target: effect_size=0.018, p_value=0.350 -> supports dropping it (no signal).

Low-signal features kept anyway (univariate weakness != uselessness):
  DriverRating   effect_size=0.017, p_value=0.368
  Sex            effect_size=0.029, p_value=0.000
  WeekOfMonth    effect_size=0.028, p_value=0.138


In [8]:
decision_log = pd.DataFrame([
    {"decision": "Drop PolicyNumber", "action": "drop column",
     "evidence_source": "live recheck: corr(PolicyNumber, Year)",
     "evidence_value": f"{policynumber_year_corr:.3f}"},
    {"decision": "Drop PolicyType", "action": "drop column (keep BasePolicy + VehicleCategory)",
     "evidence_source": "live recheck: Cramer's V vs. VehicleCategory-BasePolicy",
     "evidence_value": f"{policytype_v:.3f}"},
    {"decision": "Drop AgeOfPolicyHolder", "action": "drop column (keep numeric Age)",
     "evidence_source": "live recheck: Cramer's V vs. Age",
     "evidence_value": f"{age_policyholder_v:.3f}"},
    {"decision": "Drop RepNumber", "action": "drop column",
     "evidence_source": "feature_fraud_signal_ranking.csv",
     "evidence_value": f"p={rep_number_row['p_value']:.3f} (not significant)"},
    {"decision": "Keep DriverRating / Sex / WeekOfMonth", "action": "no change",
     "evidence_source": "feature_fraud_signal_ranking.csv",
     "evidence_value": "low univariate effect_size, kept for possible tree interactions"},
])
display(decision_log)

,decision,action,evidence_source,evidence_value
0,Drop PolicyNumber,drop column,"live recheck: corr(PolicyNumber, Year)",0.937
1,Drop PolicyType,drop column (keep BasePolicy + VehicleCategory),live recheck: Cramer's V vs. VehicleCategory-B...,1.000
2,Drop AgeOfPolicyHolder,drop column (keep numeric Age),live recheck: Cramer's V vs. Age,0.998
3,Drop RepNumber,drop column,feature_fraud_signal_ranking.csv,p=0.350 (not significant)
4,Keep DriverRating / Sex / WeekOfMonth,no change,feature_fraud_signal_ranking.csv,"low univariate effect_size, kept for possible ..."


## Step 1 — Column Decisions

Four columns are dropped, each for a specific documented reason:

- **`PolicyNumber`** — a pure sequential row identifier (1…N in chronological
  order); confirmed above to correlate strongly with `Year`, so it functions
  as a disguised timestamp. A chronological-leakage risk with no genuine
  predictive content.
- **`PolicyType`** — confirmed above to be fully redundant with
  `BasePolicy` + `VehicleCategory` (they reconstruct it exactly). The two
  clean 3-level components are kept instead of the 9-level combined field:
  informationally identical, but better-populated per category, more
  statistically reliable (`PolicyType`'s chi-square test self-flagged
  `reliable=False` in Notebook 01 due to sparse cells), and yields cleaner,
  separable SHAP explanations later (coverage type vs. vehicle class as
  distinct factors).
- **`AgeOfPolicyHolder`** — confirmed above to be near-perfectly redundant
  with numeric `Age`. The numeric version is finer-grained, so it's kept and
  the bucketed version is dropped.
- **`RepNumber`** — an operational claims-handler ID (1–16) with no
  significant association with fraud (Notebook 01, cross-checked above).
  Dropped: it has no training signal and is an internal-operations artifact,
  not a claim characteristic.

**Not dropped:** low-signal features like `DriverRating`, `Sex`,
`WeekOfMonth`. Univariate weakness is not the same as uselessness — a
gradient-boosted model may still use them in interactions, and Notebook 07's
SHAP analysis makes the final call on feature relevance. Only the identifier
and confirmed-redundant columns above are removed here.

In [9]:
DROP_COLUMNS = ["PolicyNumber", "PolicyType", "AgeOfPolicyHolder", "RepNumber"]
KEPT_IN_PLACE_OF_DROPPED = ["BasePolicy", "VehicleCategory", "Age"]


def assert_columns_present(data: pd.DataFrame, columns: list, context: str) -> None:
    """Fail loudly, with the full column list, if an expected column is missing."""
    missing = [c for c in columns if c not in data.columns]
    if missing:
        raise KeyError(f"{context}: missing column(s) {missing}. Columns present: {list(data.columns)}")


def drop_identifier_and_redundant_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Drop the four columns justified in the Step 1 markdown above."""
    assert_columns_present(data, DROP_COLUMNS, "drop_identifier_and_redundant_columns")
    return data.drop(columns=DROP_COLUMNS)


df = drop_identifier_and_redundant_columns(raw_df)

for col in DROP_COLUMNS:
    assert col not in df.columns, f"'{col}' should have been dropped but is still present."
for col in KEPT_IN_PLACE_OF_DROPPED:
    assert col in df.columns, f"'{col}' should have been kept but is missing."

print(f"Dropped: {DROP_COLUMNS}")
print(f"Remaining columns: {df.shape[1]} (was {raw_df.shape[1]})")

Dropped: ['PolicyNumber', 'PolicyType', 'AgeOfPolicyHolder', 'RepNumber']
Remaining columns: 29 (was 33)


## Step 2 — Data-Quality Fixes

Notebook 01 documented two distinct data-quality issues. They're handled
**differently on purpose**:

- **`Age == 0` (~320 rows, ~2% of the data):** these are real records with one
  invalid field, not garbage rows. Dropping them would silently shrink the
  dataset and bias the fraud rate (some are fraud cases), and a deployed
  model still has to handle a missing age at inference time. So: recode
  `Age == 0` to a genuine missing value (`NaN`) and add a boolean
  `Age_was_missing` flag — a per-row relabeling that uses no aggregate
  statistic, so it's safe to do before splitting. The actual **fill value**
  (the median) is a different matter: computing it from the full dataset
  before splitting would leak val/test information into a number used to
  transform the training data too. That fill happens inside the
  `SimpleImputer` in Step 3/4, fit on the training split only. This step only
  marks the value as missing; it does not fill it in.
- **`'0'` placeholder in `DayOfWeekClaimed` / `MonthClaimed`:** a small
  number of rows (typically 1 each) have an unrecoverable placeholder instead
  of a real claim date. Imputing a single fabricated claim-date category adds
  nothing useful; dropping 1-2 of 15,420 rows costs nothing. These rows are
  dropped now, before splitting, so every split works from the same clean row
  set.

**Left as-is:** the dataset's inconsistent-looking `Make` spellings (Accura,
Nisson, Mecedes, Porche) are stable, repeated labels in the source data, not
data-entry errors — collapsing them would be an unrequested transformation of
the raw categories, so they're kept exactly as recorded.

In [10]:
def drop_unrecoverable_placeholder_rows(data: pd.DataFrame) -> pd.DataFrame:
    """Drop rows where DayOfWeekClaimed or MonthClaimed is the '0' placeholder -
    an unrecoverable claim-date field, not worth imputing for ~1-2 rows."""
    assert_columns_present(data, ["DayOfWeekClaimed", "MonthClaimed"], "drop_unrecoverable_placeholder_rows")
    is_placeholder = (data["DayOfWeekClaimed"] == "0") | (data["MonthClaimed"] == "0")
    n_dropped = int(is_placeholder.sum())
    cleaned = data.loc[~is_placeholder].copy()
    print(f"Dropped {n_dropped} row(s) with a '0' claim-date placeholder "
          f"({100 * n_dropped / len(data):.3f}% of rows).")
    return cleaned


def flag_and_mask_invalid_age(data: pd.DataFrame) -> pd.DataFrame:
    """Recode Age == 0 to NaN and add Age_was_missing - a deterministic per-row
    relabeling that does not compute or use any aggregate statistic, so it is safe
    to apply before splitting. The fill value itself is NOT computed here."""
    assert_columns_present(data, ["Age"], "flag_and_mask_invalid_age")
    data = data.copy()
    data["Age_was_missing"] = data["Age"] == 0
    data.loc[data["Age_was_missing"], "Age"] = np.nan
    n_missing = int(data["Age_was_missing"].sum())
    print(f"Flagged {n_missing} row(s) ({100 * n_missing / len(data):.2f}%) with "
          f"Age == 0 -> Age_was_missing=True, Age set to NaN (not yet imputed).")
    return data


df = drop_unrecoverable_placeholder_rows(df)
df = flag_and_mask_invalid_age(df)

n_age_missing = int(df["Age_was_missing"].sum())
print(f"\ncleaned_df shape so far: {df.shape}")

Dropped 1 row(s) with a '0' claim-date placeholder (0.006% of rows).
Flagged 319 row(s) (2.07%) with Age == 0 -> Age_was_missing=True, Age set to NaN (not yet imputed).

cleaned_df shape so far: (15419, 30)


This produces the notebook's core intermediate artifact: `cleaned_df` — raw
data with identifiers/redundant columns dropped, unrecoverable rows dropped,
and `Age` missingness marked (but not yet filled). It is still entirely
unencoded and pre-split; it is what gets saved as `cleaned.parquet` in Step 5,
and what Step 4 splits.

## Step 3 — Encoding Plan

Every remaining feature is assigned to exactly one of four buckets. The
`ColumnTransformer` built here is **not fit yet** — fitting happens in Step 4,
after splitting, on the training data only.

- **Ordinal** — fields with a genuine, fixed rank order are mapped to integer
  ranks in that order (not arbitrary/alphabetical integers), with `'none'` (or
  the equivalent lowest bucket) mapped to rank 0: `VehiclePrice`,
  `AgeOfVehicle`, `Days_Policy_Accident`, `Days_Policy_Claim`,
  `PastNumberOfClaims`, `NumberOfSuppliments`, `NumberOfCars`,
  `AddressChange_Claim`. Note: for count-like fields (`PastNumberOfClaims`,
  `NumberOfSuppliments`) rank 0 genuinely means a zero count; for fields like
  `Days_Policy_Accident` it just means "the lowest category" — described here
  as "lowest ordinal rank," not "authentic zero count."
- **One-hot** — nominal fields with no meaningful order: `Make`, `BasePolicy`,
  `VehicleCategory`, `Fault`, `Sex`, `MaritalStatus`, `AccidentArea`,
  `PoliceReportFiled`, `WitnessPresent`, `AgentType`, and the month/day
  fields (`Month`, `DayOfWeek`, `MonthClaimed`, `DayOfWeekClaimed`).
- **Impute (numeric)** — `Age`: `SimpleImputer(strategy="median")`, fit on
  the training split only (Step 2 marked the missing values; this is where
  they actually get filled).
- **Passthrough** — already-numeric fields whose integer values already
  reflect genuine magnitude, so no encoding is needed: `WeekOfMonth`,
  `WeekOfMonthClaimed`, `Deductible`, `DriverRating`, `Year`,
  `Age_was_missing`.

**No target/mean encoding anywhere** — it leaks target information through
the encoding itself and is unnecessary given how low-cardinality every
categorical field here is.

In [11]:
ORDINAL_FEATURES: dict = {
    "VehiclePrice": ["less than 20000", "20000 to 29000", "30000 to 39000",
                      "40000 to 59000", "60000 to 69000", "more than 69000"],
    "AgeOfVehicle": ["new", "2 years", "3 years", "4 years", "5 years",
                      "6 years", "7 years", "more than 7"],
    "Days_Policy_Accident": ["none", "1 to 7", "8 to 15", "15 to 30", "more than 30"],
    "Days_Policy_Claim": ["none", "8 to 15", "15 to 30", "more than 30"],
    "PastNumberOfClaims": ["none", "1", "2 to 4", "more than 4"],
    "NumberOfSuppliments": ["none", "1 to 2", "3 to 5", "more than 5"],
    "NumberOfCars": ["1 vehicle", "2 vehicles", "3 to 4", "5 to 8", "more than 8"],
    "AddressChange_Claim": ["no change", "under 6 months", "1 year", "2 to 3 years", "4 to 8 years"],
}

NOMINAL_FEATURES = [
    "Make", "BasePolicy", "VehicleCategory", "Fault", "Sex", "MaritalStatus",
    "AccidentArea", "PoliceReportFiled", "WitnessPresent", "AgentType",
    "Month", "DayOfWeek", "MonthClaimed", "DayOfWeekClaimed",
]

NUMERIC_IMPUTE_FEATURES = ["Age"]

PASSTHROUGH_FEATURES = [
    "WeekOfMonth", "WeekOfMonthClaimed", "Deductible", "DriverRating", "Year",
    "Age_was_missing",
]

print(f"Ordinal: {len(ORDINAL_FEATURES)} | Nominal: {len(NOMINAL_FEATURES)} | "
      f"Impute: {len(NUMERIC_IMPUTE_FEATURES)} | Passthrough: {len(PASSTHROUGH_FEATURES)}")

Ordinal: 8 | Nominal: 14 | Impute: 1 | Passthrough: 6


### Validate the encoding plan against the live data

Two checks before anything is fit: every ordinal field's actual values must
be a subset of the hand-specified rank order (catching a spelling mismatch
loudly instead of silently mis-ranking categories), and every non-target
column in `cleaned_df` must be assigned to exactly one bucket (catching a
column that would otherwise be silently dropped or double-counted).

In [12]:
unexpected_categories = {}
for col, expected_categories in ORDINAL_FEATURES.items():
    observed = set(df[col].dropna().unique())
    unexpected = observed - set(expected_categories)
    if unexpected:
        unexpected_categories[col] = sorted(unexpected)

if unexpected_categories:
    raise ValueError(
        f"Ordinal category lists do not match the live data: {unexpected_categories}. "
        "Update ORDINAL_FEATURES above before proceeding - a mismatch here would "
        "silently mis-rank categories."
    )
print("All ordinal category lists match the live data.")

all_bucketed = list(ORDINAL_FEATURES) + NOMINAL_FEATURES + NUMERIC_IMPUTE_FEATURES + PASSTHROUGH_FEATURES
expected_features = set(df.columns) - {TARGET_COL}

unassigned = expected_features - set(all_bucketed)
duplicated = [c for c in set(all_bucketed) if all_bucketed.count(c) > 1]

if unassigned or duplicated:
    raise ValueError(
        f"Encoding plan does not exactly cover cleaned_df's columns. "
        f"Unassigned: {sorted(unassigned)}. Duplicated across buckets: {duplicated}."
    )
print(f"All {len(expected_features)} feature columns are assigned to exactly one bucket.")

All ordinal category lists match the live data.
All 29 feature columns are assigned to exactly one bucket.


In [13]:
def build_preprocessing_pipeline(
    ordinal_features: dict, nominal_features: list,
    numeric_impute_features: list, passthrough_features: list,
) -> ColumnTransformer:
    """Build (but do not fit) the preprocessing ColumnTransformer.

    A ColumnTransformer is itself a scikit-learn Pipeline of parallel,
    per-column-group transformers - no extra outer Pipeline wrapper is needed here
    since there is no further chained step after this. `remainder="drop"` is backed
    by the exhaustive-coverage check above, so nothing is silently discarded.
    """
    ordinal_cols = list(ordinal_features.keys())
    ordinal_categories = list(ordinal_features.values())
    return ColumnTransformer(
        transformers=[
            ("ordinal", OrdinalEncoder(categories=ordinal_categories,
                                        handle_unknown="use_encoded_value", unknown_value=-1),
             ordinal_cols),
            ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
             nominal_features),
            ("age_impute", SimpleImputer(strategy="median"), numeric_impute_features),
            ("passthrough", "passthrough", passthrough_features),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


print("build_preprocessing_pipeline() defined - not fit yet.")

build_preprocessing_pipeline() defined - not fit yet.


## Step 4 — Splitting (The Critical Part)

**Leakage rules, enforced structurally, not just by convention:**

1. The split happens on `cleaned_df` — raw cleaned data, **before** any
   encoding, imputation, or scaling is fit.
2. All transformers live in a single `ColumnTransformer`, fit on the training
   split **only**, then applied (`.transform()`, never `.fit()` again) to
   val/test. Never fit on the full dataset or on val/test.
3. The `Age` imputation median is computed from the training split only — a
   direct consequence of rule 2, verified explicitly in Step 6.
4. No resampling/SMOTE happens here — that belongs in Notebook 04, fit inside
   CV folds only.
5. The test set is only ever created, transformed with the train-fit
   pipeline, and saved. Nothing here inspects it beyond a shape/fraud-rate
   check for the audit manifest.

Two distinct, clearly labeled splits are produced from `cleaned_df` and are
never mixed:

- **Primary** — stratified 70/15/15 train/val/test on `FraudFound_P`
  (`random_state=42`). This is the split every later notebook uses for model
  selection, tuning, calibration, thresholds, and the final locked test
  evaluation.
- **Secondary** — a temporal robustness check only: train on
  `Year in {1994, 1995}`, test on `Year == 1996`. Deliberately **not**
  stratified — the natural fraud-rate drift between the two periods is the
  point (it tests whether performance holds up under distribution shift), not
  something to correct for.

### Primary split: stratified 70/15/15

In [14]:
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
assert abs((TRAIN_SIZE + VAL_SIZE + TEST_SIZE) - 1.0) < 1e-9

train_val_idx, test_idx = train_test_split(
    df.index, test_size=TEST_SIZE, stratify=df[TARGET_COL], random_state=RANDOM_STATE,
)
val_ratio_of_remainder = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=val_ratio_of_remainder,
    stratify=df.loc[train_val_idx, TARGET_COL], random_state=RANDOM_STATE,
)

train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()
test_df = df.loc[test_idx].copy()

overall_fraud_rate = df[TARGET_COL].mean() * 100
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    rate = split_df[TARGET_COL].mean() * 100
    print(f"{name:<6} n={len(split_df):>6,}  fraud={int(split_df[TARGET_COL].sum()):>4}  "
          f"fraud_rate={rate:5.2f}%")
print(f"\noverall cleaned_df fraud rate: {overall_fraud_rate:.2f}%")

train  n=10,793  fraud= 646  fraud_rate= 5.99%
val    n= 2,313  fraud= 139  fraud_rate= 6.01%
test   n= 2,313  fraud= 138  fraud_rate= 5.97%

overall cleaned_df fraud rate: 5.99%


### Secondary split: temporal robustness check (not stratified)

In [15]:
TEMPORAL_TRAIN_YEARS = [1994, 1995]
TEMPORAL_TEST_YEAR = 1996

train_temporal = df.loc[df["Year"].isin(TEMPORAL_TRAIN_YEARS)].copy()
test_temporal = df.loc[df["Year"] == TEMPORAL_TEST_YEAR].copy()

for name, split_df in [("train_temporal", train_temporal), ("test_temporal", test_temporal)]:
    rate = split_df[TARGET_COL].mean() * 100
    print(f"{name:<15} n={len(split_df):>6,}  fraud={int(split_df[TARGET_COL].sum()):>4}  "
          f"fraud_rate={rate:5.2f}%")

print(
    "\nThe fraud-rate difference between train_temporal and test_temporal is expected "
    "and is the point of this split - it is a robustness check for distribution "
    "shift, not a second primary evaluation split."
)

train_temporal  n=11,336  fraud= 710  fraud_rate= 6.26%
test_temporal   n= 4,083  fraud= 213  fraud_rate= 5.22%

The fraud-rate difference between train_temporal and test_temporal is expected and is the point of this split - it is a robustness check for distribution shift, not a second primary evaluation split.


### Fit the preprocessing pipeline — training data only

The primary pipeline (fit on primary `train_df`) is what Notebook 03 onward
actually uses, and is the one saved in Step 5. The temporal split gets its
**own**, separately-fit pipeline: reusing the primary pipeline here would be
wrong, since primary `train_df` includes some `Year == 1996` rows (the
primary split ignores `Year`) — that would let 1996 information leak into the
"trained only on 1994-95" premise the temporal check is supposed to test.
Both fits are demonstrated below to prove the pipeline runs end-to-end; the
encoded arrays themselves are not saved (only the pre-encoding splits and the
fitted pipeline are — later notebooks call `.transform()` themselves).

In [16]:
X_train = train_df.drop(columns=[TARGET_COL])
X_val = val_df.drop(columns=[TARGET_COL])
X_test = test_df.drop(columns=[TARGET_COL])

preprocessing_pipeline = build_preprocessing_pipeline(
    ORDINAL_FEATURES, NOMINAL_FEATURES, NUMERIC_IMPUTE_FEATURES, PASSTHROUGH_FEATURES,
)

X_train_encoded = preprocessing_pipeline.fit_transform(X_train)  # the ONLY fit call
X_val_encoded = preprocessing_pipeline.transform(X_val)
X_test_encoded = preprocessing_pipeline.transform(X_test)

print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_val_encoded shape:   {X_val_encoded.shape}")
print(f"X_test_encoded shape:  {X_test_encoded.shape}")
print(f"Age imputation median (from training data only): "
      f"{preprocessing_pipeline.named_transformers_['age_impute'].statistics_[0]:.1f}")

X_train_encoded shape: (10793, 93)
X_val_encoded shape:   (2313, 93)
X_test_encoded shape:  (2313, 93)
Age imputation median (from training data only): 39.0


In [17]:
X_train_temporal = train_temporal.drop(columns=[TARGET_COL])
X_test_temporal = test_temporal.drop(columns=[TARGET_COL])

temporal_pipeline = build_preprocessing_pipeline(
    ORDINAL_FEATURES, NOMINAL_FEATURES, NUMERIC_IMPUTE_FEATURES, PASSTHROUGH_FEATURES,
)
X_train_temporal_encoded = temporal_pipeline.fit_transform(X_train_temporal)  # separate fit
X_test_temporal_encoded = temporal_pipeline.transform(X_test_temporal)

print(f"X_train_temporal_encoded shape: {X_train_temporal_encoded.shape}")
print(f"X_test_temporal_encoded shape:  {X_test_temporal_encoded.shape}")
print(f"Age imputation median (temporal train only): "
      f"{temporal_pipeline.named_transformers_['age_impute'].statistics_[0]:.1f}")

X_train_temporal_encoded shape: (11336, 94)
X_test_temporal_encoded shape:  (4083, 94)
Age imputation median (temporal train only): 38.0


## Step 5 — Save to Drive

Everything produced above is persisted here in one place, matching Notebook
01's convention of printing the path of every file written. The **fitted
pipeline object itself** is saved (via `joblib`), not just transformed
arrays — later notebooks load `train.parquet` / `val.parquet` / `test.parquet`
(cleaned, unencoded) alongside this pipeline and call `.transform()`
themselves, so every notebook downstream applies the exact same fitted
transform. Only the primary pipeline is saved: it's the one every later
notebook actually uses; the temporal pipeline exists only to validate the
temporal split within this notebook.

In [18]:
def save_parquet(data: pd.DataFrame, filename: str) -> Path:
    """Save a DataFrame to the processed-data directory, record it, and print the path."""
    out_path = DATA_PROCESSED_DIR / filename
    data.to_parquet(out_path, index=True)
    saved_files.append(out_path)
    print(f"Saved -> {out_path}  ({len(data):,} rows)")
    return out_path


cleaned_path = save_parquet(df, "cleaned.parquet")
train_path = save_parquet(train_df, "train.parquet")
val_path = save_parquet(val_df, "val.parquet")
test_path = save_parquet(test_df, "test.parquet")
train_temporal_path = save_parquet(train_temporal, "train_temporal.parquet")
test_temporal_path = save_parquet(test_temporal, "test_temporal.parquet")

Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/cleaned.parquet  (15,419 rows)
Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/train.parquet  (10,793 rows)
Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/val.parquet  (2,313 rows)
Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/test.parquet  (2,313 rows)
Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/train_temporal.parquet  (11,336 rows)
Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/test_temporal.parquet  (4,083 rows)


In [19]:
pipeline_path = MODELS_DIR / "preprocessing_pipeline.joblib"
joblib.dump(preprocessing_pipeline, pipeline_path)
saved_files.append(pipeline_path)
print(f"Saved -> {pipeline_path}")

# Sanity check: reload from disk and confirm it is usable with .transform() -
# this is the exact manual check the project asked for, run automatically here.
reloaded_pipeline = joblib.load(pipeline_path)
_ = reloaded_pipeline.transform(X_val.head(3))
print("Reloaded pipeline from disk and confirmed .transform() works on it.")

Saved -> /content/drive/MyDrive/InsuranceFraudProject/models/preprocessing_pipeline.joblib
Reloaded pipeline from disk and confirmed .transform() works on it.


In [20]:
def split_stats(data: pd.DataFrame) -> dict:
    n_fraud = int(data[TARGET_COL].sum())
    return {
        "n_rows": int(len(data)),
        "n_fraud": n_fraud,
        "fraud_pct": round(100 * n_fraud / len(data), 3),
    }


split_manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "source_file": RAW_DATA_PATH.name,
    "target_column": TARGET_COL,
    "dropped_columns": DROP_COLUMNS,
    "age_was_missing_count": n_age_missing,
    "primary_split": {
        "method": "stratified train/val/test on FraudFound_P",
        "ratios": {"train": TRAIN_SIZE, "val": VAL_SIZE, "test": TEST_SIZE},
        "random_state": RANDOM_STATE,
        "splits": {
            "train": split_stats(train_df),
            "val": split_stats(val_df),
            "test": split_stats(test_df),
        },
    },
    "temporal_split": {
        "method": "Year-based robustness check, not stratified",
        "train_years": TEMPORAL_TRAIN_YEARS,
        "test_year": TEMPORAL_TEST_YEAR,
        "splits": {
            "train_temporal": split_stats(train_temporal),
            "test_temporal": split_stats(test_temporal),
        },
    },
}

manifest_path = DATA_PROCESSED_DIR / "split_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2)
saved_files.append(manifest_path)
print(f"Saved -> {manifest_path}")
display(Markdown(f"```json\n{json.dumps(split_manifest, indent=2)}\n```"))

Saved -> /content/drive/MyDrive/InsuranceFraudProject/data/processed/split_manifest.json


```json
{
  "generated_at": "2026-08-28T20:36:09.070611+00:00",
  "source_file": "fraud_oracle.csv",
  "target_column": "FraudFound_P",
  "dropped_columns": [
    "PolicyNumber",
    "PolicyType",
    "AgeOfPolicyHolder",
    "RepNumber"
  ],
  "age_was_missing_count": 319,
  "primary_split": {
    "method": "stratified train/val/test on FraudFound_P",
    "ratios": {
      "train": 0.7,
      "val": 0.15,
      "test": 0.15
    },
    "random_state": 42,
    "splits": {
      "train": {
        "n_rows": 10793,
        "n_fraud": 646,
        "fraud_pct": 5.985
      },
      "val": {
        "n_rows": 2313,
        "n_fraud": 139,
        "fraud_pct": 6.01
      },
      "test": {
        "n_rows": 2313,
        "n_fraud": 138,
        "fraud_pct": 5.966
      }
    }
  },
  "temporal_split": {
    "method": "Year-based robustness check, not stratified",
    "train_years": [
      1994,
      1995
    ],
    "test_year": 1996,
    "splits": {
      "train_temporal": {
        "n_rows": 11336,
        "n_fraud": 710,
        "fraud_pct": 6.263
      },
      "test_temporal": {
        "n_rows": 4083,
        "n_fraud": 213,
        "fraud_pct": 5.217
      }
    }
  }
}
```

## Step 6 — Assertions

These are not optional sanity prints — every check below must pass for this
notebook's output to be trusted by everything downstream. Each assertion
targets one specific leakage or correctness risk named in the steps above.

### No row-index overlap

In [21]:
assert set(train_idx).isdisjoint(val_idx), "train/val index overlap detected."
assert set(train_idx).isdisjoint(test_idx), "train/test index overlap detected."
assert set(val_idx).isdisjoint(test_idx), "val/test index overlap detected."
assert set(train_temporal.index).isdisjoint(test_temporal.index), "temporal train/test index overlap detected."
print("PASSED: no index overlap in the primary split, and none in the temporal split.")

PASSED: no index overlap in the primary split, and none in the temporal split.


### Primary splits preserve the overall fraud rate

In [22]:
FRAUD_RATE_TOLERANCE_PCT = 1.0  # percentage points

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    rate = split_df[TARGET_COL].mean() * 100
    diff = abs(rate - overall_fraud_rate)
    assert diff < FRAUD_RATE_TOLERANCE_PCT, (
        f"{name} fraud rate {rate:.2f}% drifted {diff:.2f}pt from the overall "
        f"{overall_fraud_rate:.2f}% - stratification may have failed."
    )
print(f"PASSED: train/val/test fraud rates are all within {FRAUD_RATE_TOLERANCE_PCT}pt "
      f"of the overall {overall_fraud_rate:.2f}%.")

PASSED: train/val/test fraud rates are all within 1.0pt of the overall 5.99%.


### Temporal split years are exactly as intended

In [23]:
train_temporal_years = set(train_temporal["Year"].unique())
test_temporal_years = set(test_temporal["Year"].unique())

assert train_temporal_years == set(TEMPORAL_TRAIN_YEARS), (
    f"train_temporal contains unexpected years: {train_temporal_years}"
)
assert test_temporal_years == {TEMPORAL_TEST_YEAR}, (
    f"test_temporal contains unexpected years: {test_temporal_years}"
)
print(f"PASSED: train_temporal = {sorted(int(y) for y in train_temporal_years)}, "
      f"test_temporal = {sorted(int(y) for y in test_temporal_years)}.")

PASSED: train_temporal = [1994, 1995], test_temporal = [1996].


### Dropped columns absent, kept columns present

In [24]:
for col in DROP_COLUMNS:
    assert col not in df.columns, f"'{col}' should be absent from cleaned_df."
    assert col not in train_df.columns, f"'{col}' should be absent from train_df."
for col in KEPT_IN_PLACE_OF_DROPPED:
    assert col in df.columns, f"'{col}' should be present in cleaned_df."
print(f"PASSED: {DROP_COLUMNS} absent; {KEPT_IN_PLACE_OF_DROPPED} present.")

PASSED: ['PolicyNumber', 'PolicyType', 'AgeOfPolicyHolder', 'RepNumber'] absent; ['BasePolicy', 'VehicleCategory', 'Age'] present.


### `Age_was_missing` flag is present and correctly counted

In [25]:
assert "Age_was_missing" in df.columns, "Age_was_missing column is missing."
assert int(df["Age_was_missing"].sum()) == n_age_missing, (
    "Age_was_missing sum does not match the count detected during cleaning - "
    "the flag may have been recomputed inconsistently somewhere."
)
assert df["Age"].isna().sum() == n_age_missing, (
    "Number of NaN Age values does not match Age_was_missing - they should be identical."
)
print(f"PASSED: Age_was_missing present, sums to {n_age_missing} "
      f"({100 * n_age_missing / len(df):.2f}% of {len(df):,} rows), matches Age NaN count.")

PASSED: Age_was_missing present, sums to 319 (2.07% of 15,419 rows), matches Age NaN count.


### No transformer was fit on val/test

`.transform()` must never mutate a fitted transformer's learned state. This
check captures the pipeline's fitted internals (the ordinal encoder's
`categories_` and the imputer's `statistics_`) before transforming val/test,
transforms val/test, then confirms that state is byte-for-byte unchanged —
directly catching an accidental refit rather than just asserting it by
comment.

In [26]:
categories_before = [arr.copy() for arr in preprocessing_pipeline.named_transformers_["ordinal"].categories_]
age_median_before = preprocessing_pipeline.named_transformers_["age_impute"].statistics_.copy()

_ = preprocessing_pipeline.transform(X_val)
_ = preprocessing_pipeline.transform(X_test)

categories_after = preprocessing_pipeline.named_transformers_["ordinal"].categories_
age_median_after = preprocessing_pipeline.named_transformers_["age_impute"].statistics_

assert all(np.array_equal(a, b) for a, b in zip(categories_before, categories_after)), (
    "Ordinal encoder categories changed after transforming val/test - it must never be refit."
)
assert np.array_equal(age_median_before, age_median_after), (
    "Age imputation median changed after transforming val/test - it must never be refit."
)
print("PASSED: pipeline's fitted state is unchanged after transforming val/test - "
      "confirmed it was fit on training data only.")

PASSED: pipeline's fitted state is unchanged after transforming val/test - confirmed it was fit on training data only.


All Step 6 checks passed. If any assertion above fails on a re-run, treat it
as a hard stop — do not proceed to Notebook 03 until it's resolved.

## Preprocessing & Split Summary

In [27]:
summary_md = f"""
**Columns dropped:** {", ".join(DROP_COLUMNS)} (identifiers / confirmed-redundant -
see Step 0 for the live-reverified evidence for each).

**Data-quality fixes applied:** {n_age_missing} rows had `Age == 0` -> recoded to
NaN + flagged via `Age_was_missing` (fill value deferred to the train-only fitted
pipeline); rows with a `'0'` claim-date placeholder in `DayOfWeekClaimed` or
`MonthClaimed` were dropped before splitting.

**Encoding plan:** {len(ORDINAL_FEATURES)} ordinal, {len(NOMINAL_FEATURES)} one-hot,
{len(NUMERIC_IMPUTE_FEATURES)} imputed-numeric, {len(PASSTHROUGH_FEATURES)} passthrough
features - defined as a single `ColumnTransformer`, fit exclusively on the primary
training split.

**Primary split (stratified, random_state={RANDOM_STATE}):**
train={len(train_df):,} ({split_manifest['primary_split']['splits']['train']['fraud_pct']}% fraud),
val={len(val_df):,} ({split_manifest['primary_split']['splits']['val']['fraud_pct']}% fraud),
test={len(test_df):,} ({split_manifest['primary_split']['splits']['test']['fraud_pct']}% fraud).
This is the split every later notebook uses.

**Secondary split (temporal robustness check, not stratified):**
train_temporal (Year {TEMPORAL_TRAIN_YEARS})={len(train_temporal):,}
({split_manifest['temporal_split']['splits']['train_temporal']['fraud_pct']}% fraud),
test_temporal (Year {TEMPORAL_TEST_YEAR})={len(test_temporal):,}
({split_manifest['temporal_split']['splits']['test_temporal']['fraud_pct']}% fraud).
The fraud-rate drift between them is the expected, intended signal of this check -
not something later notebooks should try to correct for.

**All Step 6 assertions passed:** no split overlap, fraud rates preserved,
column decisions enforced, `Age_was_missing` verified, and the primary pipeline's
fitted state confirmed unchanged after transforming val/test.

**Open items for Notebook 03 onward:**
1. `Logistic Regression` (Notebook 03) will need its own scaling step - not done
   here, since tree-based models don't need it and scaling on the full encoded
   matrix here would apply to val/test-derived columns too early.
2. Resampling/SMOTE comparisons (Notebook 04) must be fit inside CV folds on
   `train.parquet`, never on `val`/`test`, and never on this notebook's already-fit
   pipeline output.
3. Final feature relevance (including the low-signal features kept in Step 1) is
   deferred to Notebook 07's SHAP analysis.
"""
display(Markdown(summary_md))


**Columns dropped:** PolicyNumber, PolicyType, AgeOfPolicyHolder, RepNumber (identifiers / confirmed-redundant -
see Step 0 for the live-reverified evidence for each).

**Data-quality fixes applied:** 319 rows had `Age == 0` -> recoded to
NaN + flagged via `Age_was_missing` (fill value deferred to the train-only fitted
pipeline); rows with a `'0'` claim-date placeholder in `DayOfWeekClaimed` or
`MonthClaimed` were dropped before splitting.

**Encoding plan:** 8 ordinal, 14 one-hot,
1 imputed-numeric, 6 passthrough
features - defined as a single `ColumnTransformer`, fit exclusively on the primary
training split.

**Primary split (stratified, random_state=42):**
train=10,793 (5.985% fraud),
val=2,313 (6.01% fraud),
test=2,313 (5.966% fraud).
This is the split every later notebook uses.

**Secondary split (temporal robustness check, not stratified):**
train_temporal (Year [1994, 1995])=11,336
(6.263% fraud),
test_temporal (Year 1996)=4,083
(5.217% fraud).
The fraud-rate drift between them is the expected, intended signal of this check -
not something later notebooks should try to correct for.

**All Step 6 assertions passed:** no split overlap, fraud rates preserved,
column decisions enforced, `Age_was_missing` verified, and the primary pipeline's
fitted state confirmed unchanged after transforming val/test.

**Open items for Notebook 03 onward:**
1. `Logistic Regression` (Notebook 03) will need its own scaling step - not done
   here, since tree-based models don't need it and scaling on the full encoded
   matrix here would apply to val/test-derived columns too early.
2. Resampling/SMOTE comparisons (Notebook 04) must be fit inside CV folds on
   `train.parquet`, never on `val`/`test`, and never on this notebook's already-fit
   pipeline output.
3. Final feature relevance (including the low-signal features kept in Step 1) is
   deferred to Notebook 07's SHAP analysis.


### Files saved by this notebook

In [28]:
print("Preprocessing outputs written this run:")
for p in saved_files:
    print(f"  - {p}")

Preprocessing outputs written this run:
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/cleaned.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/train.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/val.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/test.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/train_temporal.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/test_temporal.parquet
  - /content/drive/MyDrive/InsuranceFraudProject/models/preprocessing_pipeline.joblib
  - /content/drive/MyDrive/InsuranceFraudProject/data/processed/split_manifest.json
